In [1]:
print(":ok")

:ok


In [3]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]
tavily_api_key = os.environ["TAVILY_API_KEY"]

In [4]:
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_azure_ai.utils.agents import pretty_print

model = AzureAIOpenAIApiChatModel(
    endpoint="https://grashikreddy-8785-resource.services.ai.azure.com/openai/v1/",
    credential=azure_openai_key, #if using Entra ID you can should use DefaultAzureCredential() instead
    model="gpt-5.4-mini",
    temperature=0.1
)

In [9]:
from langchain_tavily import TavilySearch

search = TavilySearch(
    max_results=5,
    topic="general",
    api_key=tavily_api_key
)

In [10]:
result = search.invoke("RAG")
print(result)

{'query': 'RAG', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'title': 'Retrieval-augmented generation - Wikipedia', 'content': 'Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM\'s pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to [...] RAG improves LLMs by incorporating information retrieval before generating responses. Unlike LLMs that rely on static training data, RAG pulls relevant text from databases, uploaded documents, or web sources. According to Ars Technica, "RAG is a way of improving LLM pe

In [11]:
tools = [search]
print(tools)

[TavilySearch(max_results=5, topic='general', api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant."
)

In [23]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Answer the following question: {question}")

messages = prompt.invoke({"question": "Rashik Reddy Gurrala in LinkedIn"}).to_messages()
result = agent.invoke({"messages": messages})
print(result["messages"][-1].content)

I found a LinkedIn profile for **Rashik Reddy Gurrala**:

- **Profile:** https://www.linkedin.com/in/rashikreddygurrala
- **Headline:** Building GenAI Solutions | LLM Fine-Tuning (QLoRA, vLLM) | RAG Architectures | MLOps | AI at Scale | LangChain | Pinecone
- **Current role:** Software Engineer at Molina Healthcare
- **Location:** Greater St. Louis, United States

If you want, I can also summarize the profile details like experience, education, or skills.


In [29]:
for token, metadata in agent.stream({"messages": messages}, stream_mode="messages"):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

I found a LinkedIn profile for **Rashik Reddy Gurrala**:

- **LinkedIn:** https://www.linkedin.com/in/rashikreddygurrala
- **Headline:** Building GenAI Solutions | LLM Fine-Tuning (QLoRA, vLLM) | RAG Architectures | MLOps | AI at Scale | LangChain | Pinecone
- **Current role:** Software Engineer at Molina Healthcare
- **Location:** Greater St. Louis, United States

If you want, I can also summarize the profile or help you find related public information.

In [34]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant.",
    checkpointer=memory
)
config = {
    "configurable": {"thread_id": "001"}
}

In [35]:
for token, metadata in agent.stream({"messages": messages}, stream_mode="messages", config=config):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

I found a LinkedIn profile that appears to match **Rashik Reddy Gurrala**:

- **LinkedIn:** https://www.linkedin.com/in/rashikreddygurrala
- **Headline:** Building GenAI Solutions | LLM Fine-Tuning (QLoRA, vLLM) | RAG Architectures | MLOps | AI at Scale | LangChain | Pinecone
- **Current role:** Software Engineer at Molina Healthcare
- **Location:** Greater St. Louis, United States

If you want, I can also summarize the profile details like experience, education, and skills.

In [36]:
for token, metadata in agent.stream({"messages": "Rashik Reddy Gurrala in LinkedIn headq says he is just a software developer at molinahealthcare"}, stream_mode="messages", config=config):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

Yes — the LinkedIn result indicates that **Rashik Reddy Gurrala** is listed as a **Software Engineer / Software Developer at Molina Healthcare**.

If you want, I can help you:
- verify the exact headline,
- compare multiple LinkedIn results,
- or format this into a clean one-line profile summary.

In [37]:
for token, metadata in agent.stream({"messages": "The from where you looked up his headline as gen AI and all"}, stream_mode="messages", config=config):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

I got that from the **search result snippet** returned by the lookup tool for his LinkedIn profile. The snippet included a detailed summary of the profile, including the headline:

> **Building GenAI Solutions | LLM Fine-Tuning (QLoRA, vLLM) | RAG Architectures | MLOps | AI at Scale | LangChain | Pinecone**

It also showed:
- **Current role:** Software Engineer
- **Company:** Molina Healthcare

So the “GenAI” part came from the **tool’s extracted profile summary**, not from me guessing.

If you want, I can help distinguish between:
1. the **public LinkedIn headline**, and  
2. the **search engine’s generated profile summary**.

In [38]:
for token, metadata in agent.stream({"messages": "distinguish between the public LinkedIn headline and the search engine’s generated profile summary"}, stream_mode="messages", config=config):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

Sure — here’s the distinction:

### 1) Public LinkedIn headline
This is the short line shown directly under a person’s name on their LinkedIn profile.  
For Rashik Reddy Gurrala, the profile result showed:

- **Software Engineer at Molina ...**  

That is the part that appears to be the actual public headline/title visible on LinkedIn.

### 2) Search engine’s generated profile summary
This is the extra text the search tool extracted and summarized from the LinkedIn page. It often combines several fields such as:
- headline
- current role
- location
- experience
- skills
- description

For this profile, the tool-generated summary included:

- **Building GenAI Solutions | LLM Fine-Tuning (QLoRA, vLLM) | RAG Architectures | MLOps | AI at Scale | LangChain | Pinecone**

That may reflect profile content, but it is not necessarily the exact headline line shown publicly on LinkedIn.

### Bottom line
- **Public headline:** “Software Engineer at Molina ...”
- **Search summary / extracted profil

In [39]:
for token, metadata in agent.stream({"messages": "How come both are very different"}, stream_mode="messages", config=config):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

That can happen for a few reasons:

1. **LinkedIn profiles have multiple fields**
   - The **headline** is the short line under the name.
   - The **About** section or other profile text can contain very different wording.
   - Search engines may mix these together in their summaries.

2. **Search tools often generate an extracted summary**
   - They don’t always show only the exact headline.
   - They may combine headline, current role, skills, and description into one snippet.

3. **Profiles can be updated separately**
   - A person may change their headline over time.
   - The search index may still show an older version, while the page now shows a newer one.

4. **Partial or inferred data**
   - Some tools infer a profile summary from visible page content.
   - That can make it look more “AI/GenAI” focused even if the actual headline is simpler.

So in this case, the difference likely means:
- **actual headline:** Software Engineer at Molina Healthcare
- **profile summary or extrac